# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalalostcode/FlyrankAI_ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

We frame this problem strictly as a **Ranking / Scoring Task**. 

Although the raw dataset provides a binary ground truth (`is_declining_label`), the downstream operational action executed by content editors is not a stochastic yes/no decision, but rather an allocation of limited manual auditing capacity. The machine learning model will predict a continuous probability score representing the likelihood and urgency of a content asset's organic performance degradation. By sorting the entire multi-client inventory by this score, we construct an optimized priority queue. This ensuring that human expertise is directed toward the highest-yield content refresh opportunities first, directly addressing the efficiency limitations of the rule-based baseline.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

# Load starter data safely across possible relative path directories
data_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

# Quick distribution check to understand target baseline characteristics
total_items = len(df)
declining_count = df['trend_direction'].value_counts().get('down', 0)
base_rate = (declining_count / total_items) * 100

print(f"--- TASK TYPE VERIFICATION ---")
print(f"Total available items for ranking: {total_items} rows")
print(f"Observed declining items: {declining_count} rows ({base_rate:.2f}% base rate)")

--- TASK TYPE VERIFICATION ---
Total available items for ranking: 30000 rows
Observed declining items: 16262 rows (54.21% base rate)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The prediction target is `is_declining_label` (Binary: 1 for declining, 0 for stable).

This label satisfies the core requirement of a robust ML framework because it is an **observed historical outcome**, not a human-defined heuristic or arbitrary rule. It is computed directly from actual traffic data tracking empirical performance loss (`trend_direction == 'down'`) over a trailing 90-day window. 

*Critical Anti-Leakage Guard:* Because `is_declining_label` is derived directly from `trend_direction` and `trend_pct`, these two columns constitute pure target leakage. They represent a classic "label trap" and must be completely isolated and removed from the feature matrix before training to prevent the model from learning deterministic shortcuts that do not exist at inference time.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Isolate and demonstrate target leakage elimination explicitly
leakage_columns = ['trend_direction', 'trend_pct']
features_df = df.drop(columns=leakage_columns, errors='ignore')

print("--- TARGET & ANTI-LEAKAGE ISOLATION ---")
print(f"Original dataframe shape: {df.shape}")
print(f"Cleaned feature matrix shape (Leakage Removed): {features_df.shape}")
print(f"Columns removed to prevent label trap: {leakage_columns}")

--- TARGET & ANTI-LEAKAGE ISOLATION ---
Original dataframe shape: (30000, 44)
Cleaned feature matrix shape (Leakage Removed): (30000, 42)
Columns removed to prevent label trap: ['trend_direction', 'trend_pct']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary evaluation success metric for this task is **Precision@50**.

Global metrics like ROC-AUC or LogLoss are highly misleading here because they evaluate model performance across the entire unranked dataset. In a real-world content operations framework, human editorial capacity is strictly capped (e.g., an editorial team can only audit and refresh 50 pages in a given work cycle). 

Precision@50 measures the exact economic utility of our model: out of the top 50 highest-scored pages recommended by the queue, what fraction are actually declining? The rule-based baseline delivers a low Precision@50 of 0.240 (meaning 38 out of 50 reviews are wasted hours). Success is defined as optimizing this ranked queue to scale Precision@50 toward our target validation rate of 0.740, directly multiplying human operational efficiency.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

# Simulate Precision@50 calculation mechanics to verify implementation viability
def compute_precision_at_k(y_true, y_scores, k=50):
    # Sort actual labels based on descending predicted scores
    order = np.argsort(y_scores)[::-1]
    y_true_sorted = np.asarray(y_true)[order]
    return np.mean(y_true_sorted[:k])

# Create a deterministic baseline check (using an arbitrary continuous column like 'ctr' as a dummy score)
dummy_scores = df['ctr'].fillna(0).values
actual_labels = (df['trend_direction'] == 'down').astype(int).values

simulated_p50 = compute_precision_at_k(actual_labels, dummy_scores, k=50)
print("--- METRIC COMPUTATION ENGINE VERIFICATION ---")
print(f"Simulated Precision@50 using single feature proxy (CTR): {simulated_p50:.4f}")

--- METRIC COMPUTATION ENGINE VERIFICATION ---
Simulated Precision@50 using single feature proxy (CTR): 0.0800


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The **Unit of Analysis** is **one unique content page per client site (`content_id` $\times$ `client_id`) evaluated over a trailing 90-day performance snapshot window**.

Each row captures the granular search performance, text metadata scale, and user engagement footprints for a specific pseudonymized asset. To make this asset vector ready for machine learning, we clean the target leakage and implement structural indicator masking for data anomalies. Specifically, `avg_position = 0` does not mean ranking position zero; it represents a complete absence of search impression data. We explicitly encode this as a distinct boolean feature flag (`missing_position_flag`) to prevent tree-split distortion.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Construct the clean, verified Unit of Analysis dataframe slice
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# 1. Clean Leakage
features_df = df.drop(columns=['trend_direction', 'trend_pct'], errors='ignore')

# 2. Handle position anomaly: avg_position = 0 means "no data"
features_df['missing_position_flag'] = (features_df['avg_position'] == 0).astype(int)

# 3. Add indicator flags for structured missingness in text data
features_df['has_word_count'] = features_df['word_count'].notna().astype(int)

# Select identifying grain, engineered robust flags, core features, and target
unit_slice = features_df[[
    'content_id', 
    'client_id', 
    'missing_position_flag', 
    'has_word_count', 
    'ctr', 
    'avg_position', 
    'is_declining_label'
]].head(5)

print("--- UNIT OF ANALYSIS DATAFRAME SLICE ---")
print(unit_slice.to_string(index=False))

--- UNIT OF ANALYSIS DATAFRAME SLICE ---
          content_id         client_id  missing_position_flag  has_word_count  ctr  avg_position  is_declining_label
content_304f48230142 client_f369cb89fc                      0               1 0.76          10.6                   1
content_a1fb4e703a9e client_4e07408562                      0               1 0.05          20.3                   1
content_9aa793d4d895 client_7f2253d7e2                      0               1 0.09          36.5                   1
content_331d6c4de07b client_19581e27de                      0               0 0.49           6.2                   0
content_d99b7a2d90ca client_3fdba35f04                      0               1 0.13          44.0                   1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A plain, fixed rule-based system (such as an `if-statement` threshold) fails catastrophically due to three primary layers of multi-client data complexity:

1. **Massive Workload and Worksite Skew:** The operational footprint per client is highly non-uniform. The largest client controls 7,008 pages, while the smallest tracks only 3 pages (with a median of 567 pages). A global rule setting a rigid threshold for traffic drops will inherently bias recommendations toward high-volume clients, completely blinding the system to severe relative degradation occurring within medium and small client ecosystems.
2. **Non-Linear Interactions & Scale Variance:** Declining pages are heavily concentrated within high-traffic brackets, showing a median impression count of 961 compared to 472 for stable pages. The relationship between organic impressions, tier shifts, and click-through rates is multi-dimensional and non-linear. Manusia tidak akan bisa menulis conditional logic manual yang secara presisi menyeimbangkan metrik-metrik ini tanpa memicu alarm false positive yang masif.
3. **Structured Anomaly Noise:** The presence of 4.02% rows with absolute zero search index indicators (`avg_position = 0`) along with systemic missing text features linked directly to `content_type` creates a messy topology. Machine learning models (specifically gradient-boosted decision trees) naturally map these complex, cross-client conditional feature behaviors and structural gaps, delivering a generalized optimization queue that a manual rule cannot match.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compute and print real numbers confirming client skew and scale variance empirically
client_counts = df['client_id'].value_counts()
imp_col = 'impressions_90d' if 'impressions_90d' in df.columns else 'impressions'

median_imp_stable = df[df['is_declining_label'] == 0][imp_col].median()
median_imp_decline = df[df['is_declining_label'] == 1][imp_col].median()
missing_pos_pct = (df['avg_position'] == 0).mean() * 100

print("--- EMPIRICAL EVIDENCE OF DATA COMPLEXITY ---")
print(f"1. Client Footprint Skew: Max = {client_counts.max()} pages vs Min = {client_counts.min()} pages (Median = {client_counts.median()})")
print(f"2. Volume Scale Mismatch: Median Impressions Stable = {median_imp_stable:.0f} vs Declining = {median_imp_decline:.0f}")
print(f"3. Structural Noise: Anomaly avg_position=0 accounts for {missing_pos_pct:.2f}% of the entire inventory")

--- EMPIRICAL EVIDENCE OF DATA COMPLEXITY ---
1. Client Footprint Skew: Max = 7008 pages vs Min = 3 pages (Median = 567.0)
2. Volume Scale Mismatch: Median Impressions Stable = 472 vs Declining = 961
3. Structural Noise: Anomaly avg_position=0 accounts for 4.02% of the entire inventory


## Self-check

Before you submit, confirm each line honestly:

- [v] Every section above is filled — markdown thinking AND the code that backs it
- [v] The notebook runs top to bottom with no errors (Runtime → Run all)
- [v] No client names, URLs, or private queries anywhere
- [v] My claims use careful words: observed, measured, directional, decision-support
- [v] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.